# Estrazione vettori

Input: arrangiamenti (.wav); drums (.mid)

Output: Tensori con X (n, f, d) Y(n, f, d) con n = numero arrangiamenti/batterie, f = numero features, d = dimensionalità.


## PARAMETRI

In [7]:
from pathlib import Path
import numpy as np
import librosa
import pretty_midi
from scipy.stats import binned_statistic

project_root = Path.cwd().parent

arrangement_types = ["bass", "full", "full_no_bass", "harm", "lead"]
splits = ["train", "test"]  

data_dir = project_root / "data"
output_dir = data_dir / "results" / "tensors"

# parametri griglia
bars = 8
beats_per_bar = 4
grid_resolution = 16
griglia = bars * grid_resolution  # 128

# parametri audio
sr = 44100
n_fft = 2048
hop_length = 512

arrangements_vectors = ["spectral_flux_low_band", "spectral_flux_mid_band", "spectral_flux_high_band", "rms", "chroma_change"]
drum_vectors = ["kick", "snare", "hihat", "openhat", "perc"]

DRUM_MAP = {
    "kick": {35,36},
    "snare": {38, 39, 40},
    "hihat": {42, 44},
    "openhat": {46},
}

## Estrazione features audio - ARRANGIAMENTO

In [8]:
def load_audio(path, bpm):
    """carica l'audio e lo taglia alle 8 battute in base al bpm"""
    duration = bars * beats_per_bar * (60.0 / bpm)
    y, _ = librosa.load(path, sr=sr, mono=True, duration=duration)
    return y

def stft_magnitude(y):
    """calcola la STFT e restituisce la matrice di magnitudo e le frequenze corrispondenti"""
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=n_fft, window="hann", center=True)
    stft = np.abs(stft)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    return stft, freqs

def spectral_flux_band(stft, freqs, low_hz, high_hz):
    """calcola il flusso spettrale per bande"""
    band_magnitude = stft[(freqs >= low_hz) & (freqs < high_hz)]
    positive_delta = np.maximum(np.diff(band_magnitude, axis=1), 0)
    return np.sqrt(np.sum(positive_delta * positive_delta, axis=0))

def spectral_flux(stft, freqs):
    spectral_flux_low = spectral_flux_band(stft, freqs, 40, 250)
    spectral_flux_mid = spectral_flux_band(stft, freqs, 250, 4000)
    spectral_flux_high = spectral_flux_band(stft, freqs, 4000, 20000)
    return spectral_flux_low, spectral_flux_mid, spectral_flux_high


def frame_times(n_frames, offset=0.0):
    """istante del centro di ogni frame; offset=0.5 per il flux e chroma_change, 0.0 per rms"""
    return (np.arange(n_frames) + offset) * hop_length / sr

def grid(values, times, bpm, stat="max"):
    """feature sulla griglia da 128 step (ogni frame va allo step più vicino)"""
    step_duration = (60.0 / bpm) * 4 / grid_resolution
    k = np.arange(griglia + 1)
    edges = (k - 0.5) * step_duration   # bordi a metà tra gli step
    out, _, _ = binned_statistic(times, values, statistic=stat, bins=edges)
    return np.nan_to_num(out, nan=0.0).astype(np.float32) # step vuoti -> 0
    
def extract_audio_features(path, bpm):
    """Estrae le feature da un file audio e le restituisce come matrice"""
    y = load_audio(path, bpm)
    stft, freqs = stft_magnitude(y)
    flux = frame_times(stft.shape[1] - 1, offset=0.5) # il flux è la differenza tra il frame i e il frame i+1

    features = []

    # spectral flux
    spectral_flux_low, spectral_flux_mid, spectral_flux_high = spectral_flux(stft, freqs)
    features.append(grid(spectral_flux_low, flux, bpm))
    features.append(grid(spectral_flux_mid, flux, bpm))
    features.append(grid(spectral_flux_high, flux, bpm))

    # rms
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length, center=True).ravel()
    features.append(grid(rms, frame_times(rms.size), bpm, stat="mean"))

    # chroma
    chroma = librosa.feature.chroma_stft(S=stft**2, sr=sr, n_fft=n_fft, norm=2, tuning=0.0) 
    chroma_change = np.linalg.norm(np.diff(chroma, axis=1), axis=0)
    features.append(grid(chroma_change, frame_times(chroma_change.size, offset=0.5), bpm))

    return np.stack(features)

## Estrazione vettori batteria - DRUMS

In [9]:
def pitch_to_drum(pitch):
    """converte un pitch MIDI in kick, snare, hihat, openhat o perc"""
    for name, pitches in DRUM_MAP.items():
        if pitch in pitches:
            return name
    return "perc"

def extract_drums(path):
    """Estrae le note della batteria da un file MIDI e le restituisce come matrice"""
    midi = pretty_midi.PrettyMIDI(path)
    tick_per_step = midi.resolution * 4 / grid_resolution # resolution = tick per semiminima

    matrix = np.zeros((5, griglia), dtype=np.float32)
    for instr in midi.instruments:
        for note in instr.notes:
            pos = midi.time_to_tick(note.start) / tick_per_step
            step = int(np.ceil(pos - 0.5)) 
            if 0 <= step < griglia:
                r = drum_vectors.index(pitch_to_drum(note.pitch))
                velocity = note.velocity / 127.0 # normalizza la velocity
                matrix[r, step] = max(matrix[r, step], velocity)
    return matrix

## Costruzione dei tensori

In [10]:
def parse_midi_name(path):
    """Estrae i metadati da un nome di file MIDI e restituisce id"""
    parts = path.stem.split("_")
    return int(parts[1])


def parse_audio_name(path):
    """Estrae i metadati da un nome di file audio e restituisce id, bpm e mood"""
    parts = path.stem.split("_")
    return int(parts[2]), int(parts[3]), parts[1]            # id, bpm, mood


def scan_arrangements(folder):
    """Scansiona una cartella di file audio e restituisce un dizionario di vettori"""
    output = {}
    for file in sorted(folder.glob("*.wav")):
        id, bpm, mood = parse_audio_name(file)
        output[id] = dict(vec=extract_audio_features(file, bpm), bpm=bpm, mood=mood)
    return output


def scan_drums(folder):
    """Scansiona una cartella di file MIDI e restituisce un dizionario di vettori di batteria"""
    output = {}
    for file in sorted(folder.rglob("*.mid")):
        id = parse_midi_name(file)
        output[id] = extract_drums(file)
    return output


def build_tensor(arrangements, drums):
    """accoppia arrangiamenti e batterie con lo stesso id e costruisce i tensori X e Y """
    ids = np.array(sorted(set(arrangements) & set(drums)))

    tensor = dict(
        X=np.stack([arrangements[i]["vec"] for i in ids]), # (n, 5, 128)
        Y=np.stack([drums[i] for i in ids]),           # (n, 5, 128)
        ids=ids,
    )
    return tensor

## Salvataggio tensori

In [11]:
for split in splits:
    drums = scan_drums(data_dir / split / "drums") 
    for arrangement_type in arrangement_types:
        arrangements = scan_arrangements(data_dir / split / "arrangements" / arrangement_type)
        tensor = build_tensor(arrangements, drums)
        output_path = output_dir / f"tensor_{split}_{arrangement_type}.npz"
        np.savez_compressed(output_path, **tensor)

## Tensore di solo batterie (serve come libreria per l'applicativo)

In [12]:

drums = scan_drums(data_dir / "train" / "drums")
ids = np.array(sorted(drums))
output_path = output_dir / f"tensor_drums.npz"
np.savez(output_path,
         ids=ids, 
         Y=np.stack([drums[i] for i in ids]), # (70, 5, 128)
         )
